## Mini Exercise - Text Processing

The feature-vector for the middle document is already provided in the BOW slide.  The Term Frequency is already provided in the start code. 

In [2]:
# Imports

import string
import pandas as pd
import nltk
import math

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

def init_nltk_resources():
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('wordnet')
    nltk.download('omw-1.4')

init_nltk_resources()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
# Starter functions

def preprocess(docs):
    clean_docs = []

    # get a lemmatizer object from NLTK
    lemma = WordNetLemmatizer()

    # get NLTK's list of stopwords
    # stop_words = stopwords.words('english')

    # create a mapper that replaces punctuations (defined
    # in string.punctuation) to an empty string
    punc = str.maketrans('', '', string.punctuation)

    for doc in docs:
        # remove punctuation
        doc_no_punc = doc.translate(punc)
        # convert all characters to lowercase (normalization)
        words = doc_no_punc.lower().split()

        # any word that is not found in NLTK's list of stopwords
        # is lemmatized to its root-form ('v' means 'verb')
        # and stored in the 'words' array
        words = [lemma.lemmatize(word, 'v')
                    for word in words]

        # join each word in our list to form back a document
        clean_docs.append(' '.join(words))

    return clean_docs

def pretty_print(feat_vecs, feat_names):
    df = pd.DataFrame(data=feat_vecs,
            index=['doc1', 'doc2', 'doc3'],
            columns=feat_names)

    print(df)

def BOW(docs):
    bow = CountVectorizer()

    # fit → learn vocabulary from docs
    # transform → convert docs into vectors
    # toarray() transforms results in a sparse matrix form to a dense matrix form
    feature_vectors = bow.fit_transform(docs).toarray()

    # returning both feature-vectors and feature-names. the
    # feature-vectors are aligned with the feature-names (vocab)
    return feature_vectors, bow.get_feature_names_out()

In [4]:
# Starter implementation

docs = [
    "The elephant sneezed at the sight of potatoes.",
    "Bats can see via echolocation. See the bat sight sneeze!",
    "Wondering, she opened the door to the studio."
]

# data cleansing
clean_docs = preprocess(docs)
print(clean_docs)

# getting feature-vectors and feature-names from
# after our BOW process
feat_vecs, feat_names = BOW(clean_docs)
print("\n TF")
pretty_print(feat_vecs, feat_names)

['the elephant sneeze at the sight of potatoes', 'bat can see via echolocation see the bat sight sneeze', 'wonder she open the door to the studio']

 TF
      at  bat  can  door  echolocation  elephant  of  open  potatoes  see  \
doc1   1    0    0     0             0         1   1     0         1    0   
doc2   0    2    1     0             1         0   0     0         0    2   
doc3   0    0    0     1             0         0   0     1         0    0   

      she  sight  sneeze  studio  the  to  via  wonder  
doc1    0      1       1       0    2   0    0       0  
doc2    0      1       1       0    1   0    1       0  
doc3    1      0       0       1    2   1    0       1  


1. Manually perform IDFs for the vocabulary of the corpus. 

In [16]:
# Function to compute IDF manually
def compute_manual_idf(feature_vecs, feature_names):
    N = len(feature_vecs)

    idfs = {}

    for idx, word in enumerate(feature_names):
        # Count docs containing word
        df = sum(1 for row in feature_vecs if row[idx] > 0)

        # Calc idf ( ln((1 + N) / (1 + DF) + 1 )
        idf = math.log((1 + N) / (1 + df)) + 1

        # Store idf in idf dict
        idfs[word] = idf

    return pd.DataFrame(idfs.items(), columns = [ 'Term', 'IDF' ])

# Implementation of IDF computation
manual_idfs = compute_manual_idf(feat_vecs, feat_names)

print(f"\nManual IDFs:\n{manual_idfs}")


Manual IDFs:
            Term       IDF
0             at  1.693147
1            bat  1.693147
2            can  1.693147
3           door  1.693147
4   echolocation  1.693147
5       elephant  1.693147
6             of  1.693147
7           open  1.693147
8       potatoes  1.693147
9            see  1.693147
10           she  1.693147
11         sight  1.287682
12        sneeze  1.287682
13        studio  1.693147
14           the  1.000000
15            to  1.693147
16           via  1.693147
17        wonder  1.693147


2. Write code to compute the IDFs for the vocabulary of the corpus

In [12]:
# Function to compute using Sklearn
def compute_sk_idf(clean_docs, feature_names):
    tfidf = TfidfVectorizer()
    tfidf.fit(clean_docs)

    return pd.DataFrame({
        'Term': feature_names,
        'IDF': tfidf.idf_
    })

# Implementation
sk_idfs = compute_sk_idf(clean_docs, feat_names)
print(f"Sklearn IDFs:\n{sk_idfs}")

Sklearn IDFs:
            Term       IDF
0             at  1.693147
1            bat  1.693147
2            can  1.693147
3           door  1.693147
4   echolocation  1.693147
5       elephant  1.693147
6             of  1.693147
7           open  1.693147
8       potatoes  1.693147
9            see  1.693147
10           she  1.693147
11         sight  1.287682
12        sneeze  1.287682
13        studio  1.693147
14           the  1.000000
15            to  1.693147
16           via  1.693147
17        wonder  1.693147


3. Verify that the results from Steps 3 and 4 are the same.

In [7]:
print("\nIDFs match?:", manual_idfs == sk_idfs)


IDFs match?: True


4. Write code to compute the TF-IDFs for the vocabulary of the corpus

In [9]:
# Generate feature vecs using TF-IDF
def compute_tfidf():
    tfidf = TfidfVectorizer() # Includes normalisation
    feature_vecs = tfidf.fit_transform(clean_docs).toarray()
    return feature_vecs, tfidf.get_feature_names_out()

feature_vecs, feature_names = compute_tfidf()
pretty_print(feature_vecs, feature_names)

            at       bat       can      door  echolocation  elephant  \
doc1  0.390669  0.000000  0.000000  0.000000      0.000000  0.390669   
doc2  0.000000  0.565558  0.282779  0.000000      0.282779  0.000000   
doc3  0.000000  0.000000  0.000000  0.367724      0.000000  0.000000   

            of      open  potatoes       see       she     sight    sneeze  \
doc1  0.390669  0.000000  0.390669  0.000000  0.000000  0.297114  0.297114   
doc2  0.000000  0.000000  0.000000  0.565558  0.000000  0.215061  0.215061   
doc3  0.000000  0.367724  0.000000  0.000000  0.367724  0.000000  0.000000   

        studio       the        to       via    wonder  
doc1  0.000000  0.461471  0.000000  0.000000  0.000000  
doc2  0.000000  0.167014  0.000000  0.282779  0.000000  
doc3  0.367724  0.434367  0.367724  0.000000  0.367724  
